# Command Safety Engine - Model Experiments

Loads the labeled command dataset, extracts the **production 20-feature vector** (`src/features/extract.py` with the real whitelist), and compares RandomForest vs GradientBoosting vs LogisticRegression via **5-fold stratified cross-validation** (accuracy, macro-F1, destructive-class recall). Ends with the deployed GradientBoosting confusion matrix on a held-out test split.

The feature extractor is imported from the package (not re-implemented here) so every number in this notebook matches the deployed model.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

sys.path.insert(0, os.path.abspath(".."))

from src.features.extract import FEATURE_NAMES, extract_features, feature_vector
from src.rules.rule_engine import load_whitelist

RANDOM_STATE = 42
N_SPLITS = 5
DESTRUCTIVE_LABEL = "destructive"

## 1. Load the labeled dataset

In [ ]:
DATA_PATH = os.path.join(os.pardir, "data", "labeled", "commands_labeled.csv")

df = pd.read_csv(DATA_PATH)
df = (
    df.dropna(subset=["command", "label"])
    .drop_duplicates(subset=["command"])
    .reset_index(drop=True)
)

print(f"Loaded {len(df)} labeled commands")
print(df["label"].value_counts().to_string())

## 2. Feature extraction (production extractor)

In [ ]:
whitelist = load_whitelist(os.path.join(os.pardir, "config", "whitelist.yaml"))
rows = []
for _, row in df.iterrows():
    feats = extract_features(str(row["command"]), whitelist=whitelist)
    rows.append(feature_vector(feats))
X = np.asarray(rows, dtype=float)
y = df["label"].values

label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(y)

print(f"Feature matrix shape: {X.shape} (20 features, {len(df)} commands)")
print(f"Feature names: {FEATURE_NAMES}")
print(f"Classes: {list(label_encoder.classes_)}")

## 3. 5-Fold Stratified Cross-Validation - RandomForest vs GradientBoosting vs LogisticRegression

Per-fold accuracy, macro-F1, and destructive-class recall, then a mean +/- std summary.

In [ ]:
models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200, max_depth=12, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=4, random_state=RANDOM_STATE
    ),
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
}

destructive_idx = None
if DESTRUCTIVE_LABEL in label_encoder.classes_:
    destructive_idx = list(label_encoder.classes_).index(DESTRUCTIVE_LABEL)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

results = []
fold_details = []

for model_name, base_model in models.items():
    fold_acc, fold_f1, fold_recall = [], [], []
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y_enc), start=1):
        X_tr, X_va = X[train_idx], X[val_idx]
        y_tr, y_va = y_enc[train_idx], y_enc[val_idx]

        if model_name == "LogisticRegression":
            scaler = StandardScaler()
            X_tr_fit = scaler.fit_transform(X_tr)
            X_va_fit = scaler.transform(X_va)
        else:
            X_tr_fit, X_va_fit = X_tr, X_va

        model = clone(base_model)
        model.fit(X_tr_fit, y_tr)
        y_pred = model.predict(X_va_fit)

        acc = accuracy_score(y_va, y_pred)
        macro = f1_score(y_va, y_pred, average="macro", zero_division=0)

        recall_val = np.nan
        if destructive_idx is not None:
            _, recall, _, _ = precision_recall_fscore_support(
                y_va, y_pred, labels=list(range(len(label_encoder.classes_))), zero_division=0
            )
            recall_val = recall[destructive_idx]

        fold_acc.append(acc)
        fold_f1.append(macro)
        fold_recall.append(recall_val)
        fold_details.append(
            {
                "model": model_name,
                "fold": fold_idx,
                "accuracy": acc,
                "macro_f1": macro,
                "destructive_recall": recall_val,
            }
        )

    results.append(
        {
            "model": model_name,
            "accuracy_mean": np.mean(fold_acc),
            "accuracy_std": np.std(fold_acc),
            "macro_f1_mean": np.mean(fold_f1),
            "macro_f1_std": np.std(fold_f1),
            "destructive_recall_mean": np.nanmean(fold_recall),
            "destructive_recall_std": np.nanstd(fold_recall),
        }
    )

fold_details_df = pd.DataFrame(fold_details)
results_df = pd.DataFrame(results).set_index("model")

print("Per-fold results:")
print(fold_details_df.round(4).to_string(index=False))
print()
print("Summary (mean +/- std across folds):")
results_df.round(4)

## 4. Deployed GradientBoosting - held-out test set + confusion matrix

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.20, stratify=y_enc, random_state=RANDOM_STATE
)

gbm_final = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, random_state=RANDOM_STATE
)
gbm_final.fit(X_train, y_train)
y_pred = gbm_final.predict(X_test)

print(f"Test Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Test Macro-F1 : {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")

cm = confusion_matrix(y_test, y_pred, labels=list(range(len(label_encoder.classes_))))
class_names = list(label_encoder.classes_)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("GradientBoosting Confusion Matrix (Held-Out Test Set)")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i, str(cm[i, j]), ha="center", va="center",
            color="white" if cm[i, j] > cm.max() / 2 else "black",
        )

fig.colorbar(im, ax=ax, label="Count")
plt.tight_layout()
plt.show()